# SAE-GINE Main Workflow — Fresh Rebuild from Raw Data

This notebook is based on the original `maincode.ipynb`, but removes hidden
dependencies on a pre-existing `processed/` folder.

### Required inputs

- `./database_300.zip`
- `./utils`

**Note**: For `database_300.zip`, please download it from https://journals.aps.org/prmaterials/supplemental/10.1103/PhysRevMaterials.8.L122201/database_300.zip

**Ref**: M. Grunert et al. Phys. Rev. Materials 8, L122201 (2024): https://doi.org/10.1103/PhysRevMaterials.8.L122201

### Graph branches preserved from the original work

- **11/16** — basic graph representation
- **19/24** — primary attribute-enriched representation used by the main SAE-GINE models
- **20/24** — auxiliary contribution-analysis representation with
  `covalent_radius_cordero`

The **20/24 contribution branch is intentionally preserved**. It is kept
separate from the primary 19/24 branch so that contribution analysis cannot
silently change the graph dimensions used by the main models.

External DFT validation is optional because the DFT archive is not generated
from `database_300.zip`.


In [ ]:
from pathlib import Path
import sys
import importlib

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Subset
from IPython.display import display

PROJECT_ROOT = Path(r"./")
DATA_ZIP = PROJECT_ROOT / "database_300.zip"
PROCESSED_DIR = PROJECT_ROOT / "processed"
TRAINING_DIR = PROCESSED_DIR / "paired_training"

# Optional analyses that require substantial extra compute or downstream data.
RUN_REPEATED_SUBSET_ROBUSTNESS = False
RUN_EXTERNAL_DFT_VALIDATION = False
RUN_POST_SCREENING_AUDITS = False

if not DATA_ZIP.exists():
    raise FileNotFoundError(
        f"Raw database not found:\n{DATA_ZIP}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_DIR.mkdir(parents=True, exist_ok=True)

import utils as TB3
import utils.features as FEAT

# Restore the canonical primary enhanced representation.
FEAT = importlib.reload(FEAT)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 100)
print("FRESH REBUILD FROM RAW DATABASE")
print("=" * 100)
print("Project root :", PROJECT_ROOT)
print("Raw database :", DATA_ZIP)
print("Processed dir:", PROCESSED_DIR)
print("Device       :", device)

dataset = TB3.CrystalOpticalDataset(
    zip_path=DATA_ZIP,
    require_any_target=True,
    require_full_spectrum=False,
    verbose=True,
)

epsR_graph, epsR_view = FEAT.build_epsR_graph_dataset(
    dataset,
    verbose=False,
)

epsI_graph, epsI_view = FEAT.build_epsI_graph_dataset(
    dataset,
    verbose=False,
)

TB3.check_paired_graph_alignment(
    epsR_graph,
    epsI_graph,
)

split = FEAT.build_fixed_paired_split(
    real_graph_ds=epsR_graph,
    imag_graph_ds=epsI_graph,
    processed_dir=PROCESSED_DIR,
    seed=2025,
    version="v1",
    overwrite=False,
)

train_idx = split["train_idx"].tolist()
val_idx = split["val_idx"].tolist()
test_idx = split["test_idx"].tolist()

epsR_train = split["epsR_train_ds"]
epsR_val = split["epsR_val_ds"]
epsR_test = split["epsR_test_ds"]

epsI_train = split["epsI_train_ds"]
epsI_val = split["epsI_val_ds"]
epsI_test = split["epsI_test_ds"]

# Primary enhanced representation used by the main SAE-GINE models.
epsR_graph_enh, epsR_view_enh = FEAT.build_epsR_graph_dataset_enhanced(
    dataset,
    verbose=False,
)

epsI_graph_enh, epsI_view_enh = FEAT.build_epsI_graph_dataset_enhanced(
    dataset,
    verbose=False,
)

TB3.check_paired_graph_alignment(
    epsR_graph_enh,
    epsI_graph_enh,
)

epsR_train_enh = Subset(epsR_graph_enh, train_idx)
epsR_val_enh = Subset(epsR_graph_enh, val_idx)
epsR_test_enh = Subset(epsR_graph_enh, test_idx)

epsI_train_enh = Subset(epsI_graph_enh, train_idx)
epsI_val_enh = Subset(epsI_graph_enh, val_idx)
epsI_test_enh = Subset(epsI_graph_enh, test_idx)

assert (
    epsR_graph[0].x.shape[-1],
    epsR_graph[0].edge_attr.shape[-1],
) == (11, 16)

assert (
    epsR_graph_enh[0].x.shape[-1],
    epsR_graph_enh[0].edge_attr.shape[-1],
) == (19, 24)

assert (
    len(epsR_train),
    len(epsR_val),
    len(epsR_test),
) == (7931, 992, 992)

print("Basic graph    :", 11, "/", 16)
print("Primary graph  :", 19, "/", 24)
print(
    "Fixed split    :",
    len(epsR_train),
    len(epsR_val),
    len(epsR_test),
)


FRESH REBUILD FROM RAW DATABASE
Project root : D:\TB3
Raw database : D:\TB3\database_300.zip
Processed dir: D:\TB3\processed
Device       : cuda
Loading dataset from: D:\TB3\database_300.zip
Dataset mode: structure -> DFT-IPA optical/electronic targets
Total JSON in zip                   : 10191
Number of valid samples             : 10189
Number of faulty samples            : 2

--- First 10 failed files ---
database_300/agm004589338.json -> No valid scalar or spectral target
database_300/agm005513815.json -> No valid scalar or spectral target

--- Quick statistics ---
direct_gap labels                   : 10189
indirect_gap labels                 : 10189
epsI_0 spectra                      : 9915
epsR_0 spectra                      : 9915
paired epsI_0 + epsR_0 spectra      : 9915
epsI_2 spectra                      : 8653
epsR_2 spectra                      : 8653
full four-channel spectra           : 8653
usable energy axis                  : 9915

Energy axis source:
energy_axis_so

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(r"./")
DATA_ZIP = PROJECT_ROOT / "database_300.zip"
PROCESSED_DIR = PROJECT_ROOT / "processed"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import utils as TB3

dataset = TB3.CrystalOpticalDataset(
    zip_path=DATA_ZIP,
    require_any_target=True,
    require_full_spectrum=False,
    verbose=True,
)

summary_df, audit_df = TB3.save_dataset_summary(
    dataset,
    out_dir=PROCESSED_DIR / "dataset_audit",
)

epsR_view, epsI_view = TB3.make_paired_spectrum_views(
    dataset,
    key_real="epsR_0",
    key_imag="epsI_0",
    verbose=True,
)

display(summary_df)

Loading dataset from: D:\TB3\database_300.zip
Dataset mode: structure -> DFT-IPA optical/electronic targets


In [ ]:
scalar_graph = TB3.build_scalar_graph_dataset(
    dataset,
    target_keys=["direct_gap", "indirect_gap"],
    attach_graph_attr=False,
)

epsR_graph, epsR_view = TB3.build_epsR_graph_dataset(
    dataset,
    verbose=False,
)

g_scalar = scalar_graph[0]
g_epsR = epsR_graph[0]

graph_summary = {
    "scalar_samples": len(scalar_graph),
    "epsR_samples": len(epsR_graph),
    "scalar_node_dim": g_scalar.x.shape[1],
    "scalar_edge_dim": g_scalar.edge_attr.shape[1],
    "epsR_node_dim": g_epsR.x.shape[1],
    "epsR_edge_dim": g_epsR.edge_attr.shape[1],
    "spectrum_points": g_epsR.y.shape[1],
    "energy_min_eV": float(g_epsR.axis_grid.min()),
    "energy_max_eV": float(g_epsR.axis_grid.max()),
}

graph_summary

In [ ]:
epsI_graph, epsI_view = TB3.build_epsI_graph_dataset(
    dataset,
    verbose=False,
)

g_epsI = epsI_graph[0]

epsI_summary = {
    "epsI_samples": len(epsI_graph),
    "epsI_node_dim": g_epsI.x.shape[1],
    "epsI_edge_dim": g_epsI.edge_attr.shape[1],
    "spectrum_points": g_epsI.y.shape[1],
    "energy_min_eV": float(g_epsI.axis_grid.min()),
    "energy_max_eV": float(g_epsI.axis_grid.max()),
}

epsI_summary

In [ ]:
assert epsR_graph.base_indices == epsI_graph.base_indices
assert len(epsR_graph) == len(epsI_graph)

paired_summary = {
    "paired_samples": len(epsR_graph),
    "epsR_target_id": int(epsR_graph[0].target_key_id[0]),
    "epsI_target_id": int(epsI_graph[0].target_key_id[0]),
}

paired_summary

In [ ]:
split = TB3.build_fixed_paired_split(
    real_graph_ds=epsR_graph,
    imag_graph_ds=epsI_graph,
    processed_dir=PROCESSED_DIR,
    seed=2025,
    version="v1",
    overwrite=False,
)

train_idx = split["train_idx"].tolist()
val_idx = split["val_idx"].tolist()
test_idx = split["test_idx"].tolist()

epsR_train = split["epsR_train_ds"]
epsR_val = split["epsR_val_ds"]
epsR_test = split["epsR_test_ds"]

epsI_train = split["epsI_train_ds"]
epsI_val = split["epsI_val_ds"]
epsI_test = split["epsI_test_ds"]

split_df = split["split_df"]

split_summary = {
    "train_samples": len(epsR_train),
    "val_samples": len(epsR_val),
    "test_samples": len(epsR_test),
    "seed": 2025,
}

split_summary


In [ ]:
import io
from contextlib import redirect_stdout

# Auxiliary contribution-analysis branch.
# This adds covalent_radius_cordero and must remain 20/24.
with redirect_stdout(io.StringIO()):
    contribution_graphs = TB3.build_enhanced_20_24_graph_splits(
        ds=dataset,
        optical_graph_epsR_ds=epsR_graph,
        optical_graph_epsI_ds=epsI_graph,
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
        verbose=False,
    )

epsR_graph_20 = contribution_graphs[
    "optical_graph_epsR_enhanced_ds"
]
epsI_graph_20 = contribution_graphs[
    "optical_graph_epsI_enhanced_ds"
]

epsR_train_20 = contribution_graphs["epsR_enhanced_train_ds"]
epsR_val_20 = contribution_graphs["epsR_enhanced_val_ds"]
epsR_test_20 = contribution_graphs["epsR_enhanced_test_ds"]

epsI_train_20 = contribution_graphs["epsI_enhanced_train_ds"]
epsI_val_20 = contribution_graphs["epsI_enhanced_val_ds"]
epsI_test_20 = contribution_graphs["epsI_enhanced_test_ds"]

assert (
    epsR_graph_20[0].x.shape[-1],
    epsR_graph_20[0].edge_attr.shape[-1],
) == (20, 24)

assert (
    epsI_graph_20[0].x.shape[-1],
    epsI_graph_20[0].edge_attr.shape[-1],
) == (20, 24)

contribution_graph_summary = {
    "epsR_samples": len(epsR_graph_20),
    "epsI_samples": len(epsI_graph_20),
    "node_features": epsR_graph_20[0].x.shape[1],
    "edge_features": epsR_graph_20[0].edge_attr.shape[1],
    "train_samples": len(epsR_train_20),
    "val_samples": len(epsR_val_20),
    "test_samples": len(epsR_test_20),
}

print("Auxiliary contribution graph: 20 / 24")
contribution_graph_summary


In [ ]:
test_model, node_weights, edge_weights = TB3.sanity_check_linear_ensemble_ue_gine(
    train_ds=epsI_train_20,
    device=device,
    hidden_dim=192,
    latent_dim=256,
    num_layers=4,
    dropout=0.05,
    top_k=10,
)

display(node_weights)
display(edge_weights)

del test_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
from torch_geometric.loader import DataLoader

loader = DataLoader(epsI_train_20, batch_size=2, shuffle=False)
batch = next(iter(loader)).to(device)

model = TB3.OpticalResponseGINE_LinearEnsembleUE(
    node_in_dim=batch.x.shape[1],
    edge_in_dim=batch.edge_attr.shape[1],
    out_dim=batch.y.shape[-1],
    hidden_dim=192,
    latent_dim=256,
    num_layers=4,
    dropout=0.05,
).to(device)

with torch.no_grad():
    pred = model(batch)

forward_check = {
    "node_features": batch.x.shape[1],
    "edge_features": batch.edge_attr.shape[1],
    "target_shape": tuple(batch.y.shape),
    "prediction_shape": tuple(pred.shape),
}

del model, batch, pred

if torch.cuda.is_available():
    torch.cuda.empty_cache()

forward_check

In [ ]:
stats = TB3.run_paired_spectrum_stats_smoke_test(
    epsR_train_ds=epsR_train,
    epsI_train_ds=epsI_train,
    device="cpu",
)

stats_summary = {}

for target, output in stats.items():
    metrics = output["metrics_debug"]

    stats_summary[target] = {
        "spectrum_points": len(output["mean_train"]),
        "mean_train": float(output["mean_train"].mean()),
        "std_train": float(output["std_train"].mean()),
        "mae_perfect": metrics["MAE_mean"],
        "rmse_perfect": metrics["RMSE_mean"],
        "cosine_perfect": metrics["CosineSim_mean"],
        "allow_negative": output["allow_negative"],
        "negative_weight": output["negative_weight"],
    }

stats_summary

# _**Baseline GINE Training for Imaginary Dielectric Spectrum ε₂(E)**_

In [ ]:
TRAINING_DIR = PROCESSED_DIR / "paired_training"
train_config = dict(
    model_class=TB3.OpticalResponseGINE,
    output_dir=TRAINING_DIR,
    device=device,
    seeds=[42, 123, 2025],
    batch_size=16,
    num_epochs=100,
    early_stop_patience=15,
    lr=5e-4,
    weight_decay=1e-4,
    hidden_dim=192,
    latent_dim=256,
    num_layers=4,
    dropout=0.10,
    peak_weight=2.0,
    smoothness_weight=0.03,
)

epsI_baseline_summary, epsI_baseline_artifacts = TB3.train_optical_gine_single_target(
    target_key="epsI_0",
    variant_name="baseline",
    train_ds=epsI_train,
    val_ds=epsI_val,
    test_ds=epsI_test,
    **train_config,
)

# **Baseline GINE Training for Real Dielectric Spectrum ε₁(E)**

In [ ]:
epsR_baseline_summary, epsR_baseline_artifacts = TB3.train_optical_gine_single_target(
    target_key="epsR_0",
    variant_name="baseline",
    train_ds=epsR_train,
    val_ds=epsR_val,
    test_ds=epsR_test,
    **train_config,
)


# _**Enhanced Universal GINE Training for Imaginary Dielectric Spectrum ε₂(E)**_

In [ ]:
from torch.utils.data import Subset
import importlib
import utils.features as FEAT

# Re-load the canonical feature module before the primary 19/24 branch.
# The 20/24 contribution branch is auxiliary and must not alter this branch.
FEAT = importlib.reload(FEAT)

epsR_graph_enh, epsR_view_enh = FEAT.build_epsR_graph_dataset_enhanced(
    dataset,
    verbose=False,
)

epsI_graph_enh, epsI_view_enh = FEAT.build_epsI_graph_dataset_enhanced(
    dataset,
    verbose=False,
)

assert epsR_graph_enh.base_indices == epsR_graph.base_indices
assert epsI_graph_enh.base_indices == epsI_graph.base_indices

assert (
    epsR_graph_enh[0].x.shape[-1],
    epsR_graph_enh[0].edge_attr.shape[-1],
) == (19, 24)

assert (
    epsI_graph_enh[0].x.shape[-1],
    epsI_graph_enh[0].edge_attr.shape[-1],
) == (19, 24)

epsR_train_enh = Subset(epsR_graph_enh, train_idx)
epsR_val_enh = Subset(epsR_graph_enh, val_idx)
epsR_test_enh = Subset(epsR_graph_enh, test_idx)

epsI_train_enh = Subset(epsI_graph_enh, train_idx)
epsI_val_enh = Subset(epsI_graph_enh, val_idx)
epsI_test_enh = Subset(epsI_graph_enh, test_idx)

enhanced_summary = {
    "node_features": epsI_graph_enh[0].x.shape[1],
    "edge_features": epsI_graph_enh[0].edge_attr.shape[1],
    "train_samples": len(epsI_train_enh),
    "val_samples": len(epsI_val_enh),
    "test_samples": len(epsI_test_enh),
}

print("Primary enhanced graph restored: 19 / 24")
enhanced_summary


### Random-initialized primary enhanced 19/24 GINE for ε₂(E)


In [ ]:
# Random-initialized primary enhanced GINE for ε2(E).
# This run was implicit in the historical notebook outputs;
# it is made explicit here so a fresh machine can rebuild Table 1.
assert epsI_train_enh[0].x.shape[-1] == 19
assert epsI_train_enh[0].edge_attr.shape[-1] == 24

epsI_enhanced_summary, epsI_enhanced_artifacts = (
    TB3.train_optical_gine_single_target(
        target_key="epsI_0",
        variant_name="enhanced",
        train_ds=epsI_train_enh,
        val_ds=epsI_val_enh,
        test_ds=epsI_test_enh,
        **train_config,
    )
)

epsI_enhanced_summary


# _**Enhanced Universal GINE Training for Real Dielectric Spectrum ε₁(E)**_

In [ ]:
epsR_enhanced_summary, epsR_enhanced_artifacts = TB3.train_optical_gine_single_target(
    target_key="epsR_0",
    variant_name="enhanced",
    train_ds=epsR_train_enh,
    val_ds=epsR_val_enh,
    test_ds=epsR_test_enh,
    **train_config,
)

# **checkpoint conversion**

In [ ]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader as PyGDataLoader

# Auxiliary basic 11/16 SSL checkpoint.
# If processed/ is absent, this checkpoint is rebuilt from the raw database.
SSL_DIR = PROCESSED_DIR / "ssl_ablation_loss_v1" / "full"
SSL_DIR.mkdir(parents=True, exist_ok=True)

SSL_CKPT_SOURCE = (
    SSL_DIR
    / "ssl_ablation_full_backbone_only_best.pt"
)

SSL_CKPT_BACKBONE = (
    SSL_DIR
    / "ssl_ablation_full_backbone_only_best_unwrapped.pt"
)

if not SSL_CKPT_BACKBONE.exists():
    print("=" * 100)
    print("BUILDING BASIC 11/16 SSL CHECKPOINT FROM DATA")
    print("=" * 100)

    basic_ssl_graph = epsI_train[0]

    basic_ssl_dims = TB3.check_ssl_graph_dims(
        basic_ssl_graph,
        expected_node_dim=11,
        expected_edge_dim=16,
    )

    basic_ssl_loader = PyGDataLoader(
        epsI_train,
        batch_size=32,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    basic_ssl_model = TB3.OpticalResponseGINE(
        node_in_dim=basic_ssl_dims["node_dim"],
        edge_in_dim=basic_ssl_dims["edge_dim"],
        out_dim=basic_ssl_dims["ssl_out_dim"],
        hidden_dim=192,
        latent_dim=256,
        num_layers=4,
        dropout=0.10,
    ).to(device)

    optimizer = torch.optim.AdamW(
        basic_ssl_model.parameters(),
        lr=1e-3,
        weight_decay=1e-5,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=80,
    )

    best_loss = float("inf")
    best_epoch = -1
    best_state = None
    history = []

    for epoch in range(1, 81):
        basic_ssl_model.train()
        losses = []

        for batch in basic_ssl_loader:
            batch = batch.to(device)

            target = TB3.graph_desc_target(batch)
            x_aug, edge_attr_aug = TB3.corrupt_graph(
                batch,
                p_node=0.15,
                noise=0.02,
            )

            pred = basic_ssl_model(
                x=x_aug,
                edge_index=batch.edge_index,
                edge_attr=edge_attr_aug,
                batch=batch.batch,
            )

            loss = F.smooth_l1_loss(
                pred,
                target,
            )

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                basic_ssl_model.parameters(),
                5.0,
            )
            optimizer.step()

            losses.append(
                float(
                    loss.detach().cpu()
                )
            )

        scheduler.step()

        epoch_loss = float(
            np.mean(losses)
        )

        history.append(
            {
                "epoch": epoch,
                "ssl_loss": epoch_loss,
                "lr": optimizer.param_groups[0]["lr"],
            }
        )

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_epoch = epoch
            best_state = copy.deepcopy(
                basic_ssl_model.state_dict()
            )

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"[Basic SSL 11/16] epoch {epoch:03d} | "
                f"loss={epoch_loss:.6f} | "
                f"best={best_loss:.6f}"
            )

    if best_state is None:
        raise RuntimeError(
            "Basic SSL pretraining failed."
        )

    backbone_state = {
        key: value
        for key, value in best_state.items()
        if not key.startswith("head.")
    }

    torch.save(
        {
            "backbone_state_dict": backbone_state,
            "best_epoch": best_epoch,
            "best_val_ssl_loss": best_loss,
            "node_in_dim": 11,
            "edge_in_dim": 16,
            "hidden_dim": 192,
            "latent_dim": 256,
            "ssl_method": "graph_descriptor_prediction",
        },
        SSL_CKPT_SOURCE,
    )

    torch.save(
        {
            "model_state_dict": backbone_state,
            "source_checkpoint": str(SSL_CKPT_SOURCE),
            "node_in_dim": 11,
            "edge_in_dim": 16,
            "hidden_dim": 192,
            "latent_dim": 256,
            "best_epoch": best_epoch,
            "best_val_ssl_loss": best_loss,
        },
        SSL_CKPT_BACKBONE,
    )

    pd.DataFrame(history).to_csv(
        SSL_DIR / "history_ssl_basic_11_16.csv",
        index=False,
    )

    del basic_ssl_model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

else:
    print(
        "Using existing basic SSL checkpoint:",
        SSL_CKPT_BACKBONE,
    )

ssl_checkpoint_summary = {
    "checkpoint": str(SSL_CKPT_BACKBONE),
    "exists": SSL_CKPT_BACKBONE.exists(),
}

ssl_checkpoint_summary


# **SSL-initialized GINE baseline on ε2(E) using 11/16 graph features**

In [ ]:
assert epsI_train[0].x.shape[1] == 11
assert epsI_train[0].edge_attr.shape[1] == 16
assert SSL_CKPT_BACKBONE.exists()

epsI_ssl_baseline_summary, epsI_ssl_baseline_artifacts = (
    TB3.train_optical_gine_single_target(
        target_key="epsI_0",
        variant_name="ssl_init_baseline_11_16",
        run_name="epsI_0_ssl_init_baseline",
        train_ds=epsI_train,
        val_ds=epsI_val,
        test_ds=epsI_test,
        pretrained_ckpt_path=SSL_CKPT_BACKBONE,
        **train_config,
    )
)


# SSL-initialized GINE baseline on ε₁(E) using 11/16 graph features

In [ ]:
assert epsR_train[0].x.shape[1] == 11
assert epsR_train[0].edge_attr.shape[1] == 16
assert SSL_CKPT_BACKBONE.exists()

epsR_ssl_baseline_summary, epsR_ssl_baseline_artifacts = (
    TB3.train_optical_gine_single_target(
        target_key="epsR_0",
        variant_name="ssl_init_baseline_11_16",
        run_name="epsR_0_ssl_init_baseline",
        train_ds=epsR_train,
        val_ds=epsR_val,
        test_ds=epsR_test,
        pretrained_ckpt_path=SSL_CKPT_BACKBONE,
        **train_config,
    )
)


# **Three-seed benchmark summary for baseline, enhanced, and SSL-init models**

In [ ]:
import pandas as pd
from IPython.display import display

# Build the auxiliary comparison directly from the runs created above.
run_tables = [
    (
        "epsI_0",
        "baseline",
        epsI_baseline_summary,
    ),
    (
        "epsI_0",
        "enhanced",
        epsI_enhanced_summary,
    ),
    (
        "epsI_0",
        "ssl_init_baseline",
        epsI_ssl_baseline_summary,
    ),
    (
        "epsR_0",
        "baseline",
        epsR_baseline_summary,
    ),
    (
        "epsR_0",
        "enhanced",
        epsR_enhanced_summary,
    ),
    (
        "epsR_0",
        "ssl_init_baseline",
        epsR_ssl_baseline_summary,
    ),
]

summary_tables = []

for target_key, variant, dataframe in run_tables:
    frame = dataframe.copy()
    frame["target_key"] = target_key
    frame["variant_group"] = variant
    summary_tables.append(frame)

all_summary = pd.concat(
    summary_tables,
    ignore_index=True,
)

benchmark_metrics = [
    "test_MAE_mean",
    "test_RMSE_mean",
    "test_CosineSim_mean",
    "test_PeakAxisError_mean",
]

benchmark_summary = (
    all_summary
    .groupby(
        [
            "target_key",
            "variant_group",
        ]
    )[benchmark_metrics]
    .agg(
        [
            "mean",
            "std",
        ]
    )
    .round(6)
)

decision_rows = []

for target_key in [
    "epsI_0",
    "epsR_0",
]:
    target_df = (
        all_summary[
            all_summary["target_key"]
            == target_key
        ]
        .groupby(
            "variant_group"
        )[benchmark_metrics]
        .mean()
    )

    baseline_mae = target_df.loc[
        "baseline",
        "test_MAE_mean",
    ]

    enhanced_mae = target_df.loc[
        "enhanced",
        "test_MAE_mean",
    ]

    ssl_mae = target_df.loc[
        "ssl_init_baseline",
        "test_MAE_mean",
    ]

    decision_rows.append(
        {
            "target_key": target_key,
            "baseline_MAE": baseline_mae,
            "enhanced_MAE": enhanced_mae,
            "ssl_init_baseline_MAE": ssl_mae,
            "ssl_gain_vs_baseline_%":
                100.0
                * (
                    baseline_mae
                    - ssl_mae
                )
                / baseline_mae,
            "enhanced_gain_vs_baseline_%":
                100.0
                * (
                    baseline_mae
                    - enhanced_mae
                )
                / baseline_mae,
            "ssl_gap_vs_enhanced_%":
                100.0
                * (
                    ssl_mae
                    - enhanced_mae
                )
                / enhanced_mae,
        }
    )

decision_summary = pd.DataFrame(
    decision_rows
).round(4)

display(benchmark_summary)
display(decision_summary)


# **SSL pretraining on enhanced 19/24 graph descriptor target**

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader as PyGDataLoader
SSL_ENHANCED_DIR = PROCESSED_DIR / "ssl_enhanced_19_24"
SSL_ENHANCED_DIR.mkdir(parents=True, exist_ok=True)
SSL_CKPT_ENHANCED = SSL_ENHANCED_DIR / "ssl_enhanced_19_24_graphdesc_best.pt"
ssl_train_ds = epsI_train_enh
g0 = ssl_train_ds[0]

ssl_dims = TB3.check_ssl_graph_dims(
    g0,
    expected_node_dim=19,
    expected_edge_dim=24,
)

ssl_loader = PyGDataLoader(
    ssl_train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

ssl_model = TB3.OpticalResponseGINE(
    node_in_dim=ssl_dims["node_dim"],
    edge_in_dim=ssl_dims["edge_dim"],
    out_dim=ssl_dims["ssl_out_dim"],
    hidden_dim=192,
    latent_dim=256,
    num_layers=4,
    dropout=0.10,
).to(device)

optimizer = torch.optim.AdamW(
    ssl_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=80,
)
best_ssl_loss = float("inf")
best_ssl_epoch = -1
ssl_history = []

start_time = time.time()

for epoch in range(1, 81):
    ssl_model.train()
    epoch_losses = []

    for batch in ssl_loader:
        batch = batch.to(device)

        target = TB3.graph_desc_target(batch)
        x_aug, edge_attr_aug = TB3.corrupt_graph(
            batch,
            p_node=0.15,
            noise=0.02,
        )

        pred = ssl_model(
            x=x_aug,
            edge_index=batch.edge_index,
            edge_attr=edge_attr_aug,
            batch=batch.batch,
        )

        loss = F.smooth_l1_loss(pred, target)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ssl_model.parameters(), 5.0)
        optimizer.step()

        epoch_losses.append(float(loss.detach().cpu()))

    scheduler.step()

    ssl_loss = float(np.mean(epoch_losses))
    ssl_history.append(
        {
            "epoch": epoch,
            "ssl_loss": ssl_loss,
            "lr": optimizer.param_groups[0]["lr"],
        }
    )
    if ssl_loss < best_ssl_loss:
        best_ssl_loss = ssl_loss
        best_ssl_epoch = epoch

        torch.save(
            {
                "model_state_dict": ssl_model.state_dict(),
                "best_epoch": best_ssl_epoch,
                "best_ssl_loss": best_ssl_loss,
                "node_in_dim": ssl_dims["node_dim"],
                "edge_in_dim": ssl_dims["edge_dim"],
                "ssl_out_dim": ssl_dims["ssl_out_dim"],
                "ssl_method": "enhanced_graph_descriptor_prediction",
            },
            SSL_CKPT_ENHANCED,
        )

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"[SSL enhanced 19/24] epoch {epoch:03d} | "
            f"loss={ssl_loss:.6f} | best={best_ssl_loss:.6f}"
        )
ssl_history_df = pd.DataFrame(ssl_history)
ssl_history_df.to_csv(
    SSL_ENHANCED_DIR / "history_ssl_enhanced_19_24_graphdesc.csv",
    index=False,
)
ssl_enhanced_summary = {
    "train_samples": len(ssl_train_ds),
    "node_features": ssl_dims["node_dim"],
    "edge_features": ssl_dims["edge_dim"],
    "ssl_out_dim": ssl_dims["ssl_out_dim"],
    "best_epoch": best_ssl_epoch,
    "best_ssl_loss": best_ssl_loss,
    "checkpoint": str(SSL_CKPT_ENHANCED),
    "time_min": (time.time() - start_time) / 60,
}
ssl_enhanced_summary


# **Extract enhanced SSL backbone checkpoint for fine-tuning**

In [ ]:
SSL_CKPT_ENHANCED_BACKBONE = SSL_CKPT_ENHANCED.with_name(
    "ssl_enhanced_19_24_backbone_only_for_finetune.pt"
)

ssl_ckpt = torch.load(SSL_CKPT_ENHANCED, map_location="cpu")
ssl_state = ssl_ckpt["model_state_dict"]

backbone_state = {
    k: v for k, v in ssl_state.items()
    if not k.startswith("head.")
}

torch.save(
    {
        "model_state_dict": backbone_state,
        "source_checkpoint": str(SSL_CKPT_ENHANCED),
        "best_epoch": ssl_ckpt["best_epoch"],
        "best_ssl_loss": ssl_ckpt["best_ssl_loss"],
        "node_in_dim": 19,
        "edge_in_dim": 24,
        "ssl_method": "enhanced_graphdesc_backbone_only",
    },
    SSL_CKPT_ENHANCED_BACKBONE,
)

checkpoint_summary = {
    "checkpoint": str(SSL_CKPT_ENHANCED_BACKBONE),
    "num_tensors": len(backbone_state),
}

checkpoint_summary

# **SSL-init enhanced 19/24 fine-tuning for ε₂(E)**

In [ ]:
from pathlib import Path
import torch

PROCESSED_DIR = Path(r"D:\TB3\processed")
OUTPUT_DIR = PROCESSED_DIR / "paired_training"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE =", DEVICE)

In [ ]:
# Config benchmark optical GINE
train_config = dict(
    model_class=TB3.OpticalResponseGINE,
    output_dir=OUTPUT_DIR,
    device=DEVICE,

    seeds=(42, 123, 2025),
    batch_size=16,
    num_epochs=100,
    early_stop_patience=15,

    lr=5e-4,
    weight_decay=1e-4,

    hidden_dim=192,
    latent_dim=256,
    num_layers=4,
    dropout=0.10,

    peak_weight=2.0,
    smoothness_weight=0.03,

    sched_factor=0.5,
    sched_patience=4,
    min_lr=1e-6,
    clip_grad_norm=5.0,

    num_workers=0,
)
print("train_config ready")

In [ ]:
import torch
from torch.utils.data import Dataset

class DropLastNodeFeatureDataset(Dataset):
    """
    Convert 20/24 graph dataset to 19/24 by dropping the last node feature.
    This is used to avoid mixing contribution-analysis graph with main benchmark graph.
    """
    def __init__(self, base_ds):
        self.base_ds = base_ds

    def __len__(self):
        return len(self.base_ds)

    def __getitem__(self, idx):
        data = self.base_ds[idx].clone()

        if data.x.shape[1] == 20:
            data.x = data.x[:, :19]

        return data

# Convert current 20/24 enhanced splits to 19/24 benchmark splits
epsI_train_enh_19 = DropLastNodeFeatureDataset(epsI_train_enh)
epsI_val_enh_19   = DropLastNodeFeatureDataset(epsI_val_enh)
epsI_test_enh_19  = DropLastNodeFeatureDataset(epsI_test_enh)

epsR_train_enh_19 = DropLastNodeFeatureDataset(epsR_train_enh)
epsR_val_enh_19   = DropLastNodeFeatureDataset(epsR_val_enh)
epsR_test_enh_19  = DropLastNodeFeatureDataset(epsR_test_enh)

# Check
print("epsI train:", epsI_train_enh_19[0].x.shape[1], epsI_train_enh_19[0].edge_attr.shape[1])
print("epsR train:", epsR_train_enh_19[0].x.shape[1], epsR_train_enh_19[0].edge_attr.shape[1])

assert epsI_train_enh_19[0].x.shape[1] == 19
assert epsI_train_enh_19[0].edge_attr.shape[1] == 24
assert epsR_train_enh_19[0].x.shape[1] == 19
assert epsR_train_enh_19[0].edge_attr.shape[1] == 24

In [ ]:
ssl_enhanced_config = train_config.copy()

ssl_enhanced_config.update(
    lr=3e-4,
    weight_decay=5e-5,
    dropout=0.05,
)

assert epsI_train_enh_19[0].x.shape[1] == 19
assert epsI_train_enh_19[0].edge_attr.shape[1] == 24
assert SSL_CKPT_ENHANCED_BACKBONE.exists()

epsI_ssl_enhanced_summary, epsI_ssl_enhanced_artifacts = (
    TB3.train_optical_gine_single_target(
        target_key="epsI_0",
        variant_name="ssl_init_enhanced_19_24_d005_wd5e5",
        run_name="epsI_0_ssl_init_enhanced_19_24_d005_wd5e5_3seed",
        train_ds=epsI_train_enh_19,
        val_ds=epsI_val_enh_19,
        test_ds=epsI_test_enh_19,
        pretrained_ckpt_path=SSL_CKPT_ENHANCED_BACKBONE,
        **ssl_enhanced_config,
    )
)


In [ ]:
# Rebuild true 20/24 graph splits for LinearEnsembleUE
def unwrap_graph_dataset(obj):
    """
    Recover the underlying graph dataset from Subset / wrapper.
    """
    if hasattr(obj, "base_indices"):
        return obj
    if hasattr(obj, "dataset"):
        return unwrap_graph_dataset(obj.dataset)
    if hasattr(obj, "base_ds"):
        return unwrap_graph_dataset(obj.base_ds)
    raise TypeError(f"Cannot recover graph dataset from object type: {type(obj)}")


# Recover reference graph datasets for alignment check
epsR_graph_ref = unwrap_graph_dataset(epsR_train_enh)
epsI_graph_ref = unwrap_graph_dataset(epsI_train_enh)

# Recover raw crystal dataset
if hasattr(epsI_graph_ref, "base_dataset"):
    raw_dataset_for_20 = epsI_graph_ref.base_dataset
else:
    raw_dataset_for_20 = dataset   # fallback nếu biến dataset của bạn vẫn còn trong notebook

# Build 20/24 graph splits
outputs_20 = TB3.build_enhanced_20_24_graph_splits(
    ds=raw_dataset_for_20,
    optical_graph_epsR_ds=epsR_graph_ref,
    optical_graph_epsI_ds=epsI_graph_ref,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
    verbose=True,
)

# Rename to short variables
epsI_train_20 = outputs_20["epsI_enhanced_train_ds"]
epsI_val_20   = outputs_20["epsI_enhanced_val_ds"]
epsI_test_20  = outputs_20["epsI_enhanced_test_ds"]

epsR_train_20 = outputs_20["epsR_enhanced_train_ds"]
epsR_val_20   = outputs_20["epsR_enhanced_val_ds"]
epsR_test_20  = outputs_20["epsR_enhanced_test_ds"]

print("epsI 20/24:", epsI_train_20[0].x.shape[1], epsI_train_20[0].edge_attr.shape[1])
print("epsR 20/24:", epsR_train_20[0].x.shape[1], epsR_train_20[0].edge_attr.shape[1])

assert epsI_train_20[0].x.shape[1] == 20
assert epsI_train_20[0].edge_attr.shape[1] == 24
assert epsR_train_20[0].x.shape[1] == 20
assert epsR_train_20[0].edge_attr.shape[1] == 24

print("True 20/24 graph splits are ready.")

In [ ]:
# Enhanced 19/24 SSL checkpoint created earlier in this notebook.
SSL_CKPT_ENHANCED_BACKBONE = Path(
    SSL_CKPT_ENHANCED_BACKBONE
)

print(
    "SSL checkpoint:",
    SSL_CKPT_ENHANCED_BACKBONE,
)
print(
    "Exists:",
    SSL_CKPT_ENHANCED_BACKBONE.exists(),
)

assert SSL_CKPT_ENHANCED_BACKBONE.exists(), (
    "Enhanced SSL backbone checkpoint was not created."
)


In [ ]:
linear_ue_config = train_config.copy()

linear_ue_config.update(
    model_class=TB3.OpticalResponseGINE_LinearEnsembleUE,
    lr=3e-4,
    weight_decay=5e-5,
    dropout=0.05,
)

assert epsI_train_20[0].x.shape[1] == 20
assert epsI_train_20[0].edge_attr.shape[1] == 24

epsI_ssl_linear_summary, epsI_ssl_linear_artifacts = (
    TB3.train_optical_gine_single_target(
        target_key="epsI_0",
        variant_name="ssl_init_linear_ue_20_24",
        run_name="epsI_0_ssl_init_linear_ue_20_24",
        train_ds=epsI_train_20,
        val_ds=epsI_val_20,
        test_ds=epsI_test_20,
        pretrained_ckpt_path=SSL_CKPT_ENHANCED_BACKBONE,
        **linear_ue_config,
    )
)

epsI_ssl_linear_summary


# **SSL-init enhanced 19/24 fine-tuning for ε₁(E)**

In [ ]:
from torch.utils.data import Dataset
class DropLastNodeFeatureDataset(Dataset):
    """
    Convert 20/24 contribution-analysis graph to 19/24 main benchmark graph
    by dropping the last node feature.
    If input is already 19/24, keep it unchanged.
    """
    def __init__(self, base_ds):
        self.base_ds = base_ds

    def __len__(self):
        return len(self.base_ds)

    def __getitem__(self, idx):
        data = self.base_ds[idx].clone()

        node_dim = data.x.shape[1]
        edge_dim = data.edge_attr.shape[1]

        if edge_dim != 24:
            raise ValueError(f"Unexpected edge feature dimension: {edge_dim}. Expected 24.")

        if node_dim == 20:
            data.x = data.x[:, :19]
        elif node_dim == 19:
            pass
        else:
            raise ValueError(f"Unexpected node feature dimension: {node_dim}. Expected 19 or 20.")

        return data
# Make safe 19/24 datasets for epsR
epsR_train_enh_19 = DropLastNodeFeatureDataset(epsR_train_enh)
epsR_val_enh_19   = DropLastNodeFeatureDataset(epsR_val_enh)
epsR_test_enh_19  = DropLastNodeFeatureDataset(epsR_test_enh)

print("epsR train 19/24:", epsR_train_enh_19[0].x.shape[1], epsR_train_enh_19[0].edge_attr.shape[1])
print("epsR val   19/24:", epsR_val_enh_19[0].x.shape[1], epsR_val_enh_19[0].edge_attr.shape[1])
print("epsR test  19/24:", epsR_test_enh_19[0].x.shape[1], epsR_test_enh_19[0].edge_attr.shape[1])

assert epsR_train_enh_19[0].x.shape[1] == 19
assert epsR_train_enh_19[0].edge_attr.shape[1] == 24

In [ ]:
from pathlib import Path
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PROCESSED_DIR = Path(r"D:\TB3\processed")
OUTPUT_DIR = PROCESSED_DIR / "paired_training"

try:
    train_config
    print("train_config already exists.")
except NameError:
    print("train_config not found. Recreating train_config...")

    train_config = dict(
        model_class=TB3.OpticalResponseGINE,
        output_dir=OUTPUT_DIR,
        device=DEVICE,

        seeds=(42, 123, 2025),
        batch_size=16,
        num_epochs=100,
        early_stop_patience=15,

        lr=5e-4,
        weight_decay=1e-4,

        hidden_dim=192,
        latent_dim=256,
        num_layers=4,
        dropout=0.10,

        peak_weight=2.0,
        smoothness_weight=0.03,

        sched_factor=0.5,
        sched_patience=4,
        min_lr=1e-6,

        clip_grad_norm=5.0,
        num_workers=0,
    )
# SSL enhanced 19/24 finetuning config
ssl_enhanced_config = train_config.copy()
ssl_enhanced_config.update(
    model_class=TB3.OpticalResponseGINE,
    lr=3e-4,
    weight_decay=5e-5,
    dropout=0.05,
)

print("DEVICE:", DEVICE)
print("ssl_enhanced_config ready.")
print("model_class:", ssl_enhanced_config["model_class"])
print("lr:", ssl_enhanced_config["lr"])
print("weight_decay:", ssl_enhanced_config["weight_decay"])
print("dropout:", ssl_enhanced_config["dropout"])

In [ ]:
assert epsR_train_enh_19[0].x.shape[1] == 19
assert epsR_train_enh_19[0].edge_attr.shape[1] == 24
assert SSL_CKPT_ENHANCED_BACKBONE.exists()

epsR_ssl_enhanced_summary, epsR_ssl_enhanced_artifacts = (
    TB3.train_optical_gine_single_target(
        target_key="epsR_0",
        variant_name="ssl_init_enhanced_19_24_d005_wd5e5",
        run_name="epsR_0_ssl_init_enhanced_19_24_d005_wd5e5_3seed",
        train_ds=epsR_train_enh_19,
        val_ds=epsR_val_enh_19,
        test_ds=epsR_test_enh_19,
        pretrained_ckpt_path=SSL_CKPT_ENHANCED_BACKBONE,
        **ssl_enhanced_config,
    )
)

epsR_ssl_enhanced_summary


# **SSL-init Linear Ensemble UE-GINE 20/24 for ε₁(E)**

In [ ]:
linear_ue_config = train_config.copy()

linear_ue_config.update(
    model_class=TB3.OpticalResponseGINE_LinearEnsembleUE,
    lr=3e-4,
    weight_decay=5e-5,
    dropout=0.05,
)

assert epsR_train_20[0].x.shape[1] == 20
assert epsR_train_20[0].edge_attr.shape[1] == 24
assert SSL_CKPT_ENHANCED_BACKBONE.exists()

epsR_ssl_linear_summary, epsR_ssl_linear_artifacts = (
    TB3.train_optical_gine_single_target(
        target_key="epsR_0",
        variant_name="ssl_init_linear_ue_20_24",
        run_name="epsR_0_ssl_init_linear_ue_20_24",
        train_ds=epsR_train_20,
        val_ds=epsR_val_20,
        test_ds=epsR_test_20,
        pretrained_ckpt_path=SSL_CKPT_ENHANCED_BACKBONE,
        **linear_ue_config,
    )
)

epsR_ssl_linear_summary


In [ ]:
# Export the learned 20/24 descriptor contribution probabilities.
contribution_outputs = (
    TB3.export_linear_ensemble_contribution_weights_cleanfig(
        epsI_df=epsI_ssl_linear_summary,
        epsR_df=epsR_ssl_linear_summary,
        epsI_ds=epsI_train_20,
        epsR_ds=epsR_train_20,
        root=TRAINING_DIR,
        device=device,
        model_class=TB3.OpticalResponseGINE_LinearEnsembleUE,
        top_k=12,
        show=True,
    )
)

node_contribution = contribution_outputs["node_meanstd"]
edge_contribution = contribution_outputs["edge_meanstd"]

display(
    node_contribution
    .groupby("target_key")
    .head(10)
    .reset_index(drop=True)
)

display(
    edge_contribution
    .groupby("target_key")
    .head(10)
    .reset_index(drop=True)
)


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

def nice(s):
    return str(s).replace("abs_delta_X", "|Δχ|") \
                 .replace("abs_delta_ionization", "|ΔEion|") \
                 .replace("radius_sum", "ri + rj") \
                 .replace("abs_delta_radius", "|Δr|") \
                 .replace("distance_rbf_", "Distance RBF ") \
                 .replace("_", " ").title()

def top(df, target):
    d = (
        df[df["target_key"].eq(target)]
        .sort_values("weight_mean", ascending=False)
        .head(12)
        .iloc[::-1]
    )
    return d["feature"].map(nice), d["weight_mean"]

fig, ax = plt.subplots(2, 2, figsize=(12, 8), dpi=300)

items = [
    (ax[0,0], node_contribution, "epsI_0", r"a) $\varepsilon_2(E)$ node descriptors"),
    (ax[0,1], edge_contribution, "epsI_0", r"b) $\varepsilon_2(E)$ edge descriptors"),
    (ax[1,0], node_contribution, "epsR_0", r"c) $\varepsilon_1(E)$ node descriptors"),
    (ax[1,1], edge_contribution, "epsR_0", r"d) $\varepsilon_1(E)$ edge descriptors"),
]

for a, df, target, title in items:
    lab, val = top(df, target)
    a.barh(lab, val, color="#1f77b4")
    a.set_title(title, loc="left")
    a.set_xlabel("Contribution probability")
    a.spines["top"].set_visible(False)
    a.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
assert epsI_train_20[0].x.shape[1] == 20
assert epsI_train_20[0].edge_attr.shape[1] == 24

epsI_scratch_linear_summary, epsI_scratch_linear_artifacts = TB3.train_optical_gine_single_target(
    target_key="epsI_0",
    variant_name="scratch_linear_ue_20_24_epsI",
    train_ds=epsI_train_20,
    val_ds=epsI_val_20,
    test_ds=epsI_test_20,
    **linear_ue_config,
)

epsI_scratch_linear_summary


# **#  Scratch Linear Ensemble UE-GINE 20/24 for ε₁(E)**

In [ ]:
assert epsR_train_20[0].x.shape[1] == 20
assert epsR_train_20[0].edge_attr.shape[1] == 24

epsR_scratch_linear_summary, epsR_scratch_linear_artifacts = TB3.train_optical_gine_single_target(
    target_key="epsR_0",
    variant_name="scratch_linear_ue_20_24_epsR",
    train_ds=epsR_train_20,
    val_ds=epsR_val_20,
    test_ds=epsR_test_20,
    **linear_ue_config,
)

epsR_scratch_linear_summary


# **Compare scratch and SSL-init Linear Ensemble UE-GINE 20/24**

In [ ]:
import pandas as pd
from IPython.display import display

def get_run_df(*names):
    for name in names:
        if name in globals():
            return globals()[name]
    raise NameError(f"Missing variables: {names}")

epsI_ssl_df = get_run_df("epsI_ssl_linear_summary", "epsI_ssl_linear_df")
epsR_ssl_df = get_run_df("epsR_ssl_linear_summary", "epsR_ssl_linear_df")
epsI_scratch_df = get_run_df("epsI_scratch_linear_summary", "epsI_scratch_linear_df")
epsR_scratch_df = get_run_df("epsR_scratch_linear_summary", "epsR_scratch_linear_df")

runs = [
    ("epsI_0", "ε₂(E) / epsI₀", "SSL-init Linear Ensemble UE-GINE", "SSL-init", epsI_ssl_df),
    ("epsI_0", "ε₂(E) / epsI₀", "Scratch Linear Ensemble UE-GINE", "Scratch", epsI_scratch_df),
    ("epsR_0", "ε₁(E) / epsR₀", "SSL-init Linear Ensemble UE-GINE", "SSL-init", epsR_ssl_df),
    ("epsR_0", "ε₁(E) / epsR₀", "Scratch Linear Ensemble UE-GINE", "Scratch", epsR_scratch_df),
]

rows = []

for target_key, target_label, model_name, init_type, df in runs:
    rows.append(
        {
            "target_key": target_key,
            "Target": target_label,
            "Model": model_name,
            "Init": init_type,
            "Graph": "20/24",
            "MAE_mean": df["test_MAE_mean"].mean(),
            "MAE_std": df["test_MAE_mean"].std(ddof=1),
            "RMSE_mean": df["test_RMSE_mean"].mean(),
            "RMSE_std": df["test_RMSE_mean"].std(ddof=1),
            "Cosine_mean": df["test_CosineSim_mean"].mean(),
            "Cosine_std": df["test_CosineSim_mean"].std(ddof=1),
        }
    )
summary = pd.DataFrame(rows)

paper_table = pd.DataFrame(
    {
        "Target": summary["Target"],
        "Model": summary["Model"],
        "Init": summary["Init"],
        "Graph": summary["Graph"],
        "MAE ↓": summary.apply(lambda r: f"{r['MAE_mean']:.3f} ± {r['MAE_std']:.3f}", axis=1),
        "RMSE ↓": summary.apply(lambda r: f"{r['RMSE_mean']:.3f} ± {r['RMSE_std']:.3f}", axis=1),
        "Cosine ↑": summary.apply(lambda r: f"{r['Cosine_mean']:.3f} ± {r['Cosine_std']:.3f}", axis=1),
    }
)

gain_rows = []

for target_key, target_label in [("epsI_0", "ε₂(E) / epsI₀"), ("epsR_0", "ε₁(E) / epsR₀")]:
    scratch = summary[(summary["target_key"] == target_key) & (summary["Init"] == "Scratch")].iloc[0]
    ssl = summary[(summary["target_key"] == target_key) & (summary["Init"] == "SSL-init")].iloc[0]

    gain_rows.append(
        {
            "target_key": target_key,
            "target_label": target_label,
            "scratch_test_MAE": scratch["MAE_mean"],
            "ssl_init_test_MAE": ssl["MAE_mean"],
            "MAE_reduction_abs": scratch["MAE_mean"] - ssl["MAE_mean"],
            "MAE_reduction_percent": (scratch["MAE_mean"] - ssl["MAE_mean"]) / scratch["MAE_mean"] * 100,
            "scratch_test_RMSE": scratch["RMSE_mean"],
            "ssl_init_test_RMSE": ssl["RMSE_mean"],
            "RMSE_reduction_abs": scratch["RMSE_mean"] - ssl["RMSE_mean"],
            "scratch_test_Cosine": scratch["Cosine_mean"],
            "ssl_init_test_Cosine": ssl["Cosine_mean"],
            "Cosine_gain_abs": ssl["Cosine_mean"] - scratch["Cosine_mean"],
        }
    )
gain_table = pd.DataFrame(gain_rows).round(6)

display(paper_table)
display(gain_table)

# **Top feature contributions from Linear Ensemble UE-GINE 20/24**

In [ ]:
import pandas as pd
from IPython.display import display

def pick_df(*names):
    for name in names:
        if name in globals():
            return globals()[name]
    raise NameError(f"Missing variables: {names}")

node_df = pick_df("node_contribution", "node_meanstd")
edge_df = pick_df("edge_contribution", "edge_meanstd")

target_labels = {
    "epsR_0": "ε₁(E) / epsR₀",
    "epsI_0": "ε₂(E) / epsI₀",
}

def make_top_table(df, feature_type, top_k=5):
    rows = []

    for target_key in ["epsR_0", "epsI_0"]:
        sub = (
            df[df["target_key"] == target_key]
            .sort_values("weight_mean", ascending=False)
            .head(top_k)
            .reset_index(drop=True)
        )

        for rank, row in sub.iterrows():
            rows.append({
                "Target": target_labels[target_key],
                "Model": "SSL-init Linear Ensemble UE-GINE",
                "Rank": rank + 1,
                "Feature type": feature_type,
                "Feature": row["feature"],
                "Contribution weight": f"{row['weight_mean']:.4f} ± {row['weight_std']:.4f}",
            })

    return pd.DataFrame(rows)

node_top5 = make_top_table(node_df, "node")
edge_top5 = make_top_table(edge_df, "edge")

display(node_top5)
display(edge_top5)

# LOAD. Reload utility functions for SSL-gain dot plot

In [ ]:
import sys
import importlib
from pathlib import Path

PROJECT_ROOT = Path(r"./")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import utils as TB3

TB3 = importlib.reload(TB3)

assert hasattr(TB3, "plot_linear_ensemble_ssl_gain_dotplot_final")

utils_status = {
    "loaded_from": TB3.__file__,
    "has_dotplot": True,
}

utils_status


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
import utils as TB3

ROOT = PROCESSED_DIR / "paired_training"
OUT_DIR = ROOT / "linear_ensemble_contribution_20_24"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SEEDS = {
    42,
    123,
    2025,
}

runs = {
    "epsI_0": (
        "epsI_0_scratch_linear_ue_20_24_epsI",
        "epsI_0_ssl_init_linear_ue_20_24",
        r"$\varepsilon_2(E)$",
    ),
    "epsR_0": (
        "epsR_0_scratch_linear_ue_20_24_epsR",
        "epsR_0_ssl_init_linear_ue_20_24",
        r"$\varepsilon_1(E)$",
    ),
}

rows = []

for target_key, (
    scratch_dir,
    ssl_dir,
    label,
) in runs.items():
    scratch_files = sorted(
        (
            ROOT
            / scratch_dir
        ).glob(
            "summary_optical_*.csv"
        )
    )

    ssl_files = sorted(
        (
            ROOT
            / ssl_dir
        ).glob(
            "summary_optical_*.csv"
        )
    )

    if len(scratch_files) != 1:
        raise RuntimeError(
            f"{target_key}: expected one scratch summary, "
            f"found {len(scratch_files)}."
        )

    if len(ssl_files) != 1:
        raise RuntimeError(
            f"{target_key}: expected one SSL summary, "
            f"found {len(ssl_files)}."
        )

    scratch = pd.read_csv(
        scratch_files[0]
    )

    ssl = pd.read_csv(
        ssl_files[0]
    )

    if (
        set(
            scratch["seed"].astype(int)
        )
        != EXPECTED_SEEDS
        or set(
            ssl["seed"].astype(int)
        )
        != EXPECTED_SEEDS
    ):
        raise ValueError(
            f"{target_key}: unexpected seeds found."
        )

    scratch_mae = scratch[
        "test_MAE_mean"
    ]

    ssl_mae = ssl[
        "test_MAE_mean"
    ]

    rows.append(
        {
            "target_key":
                target_key,
            "Target":
                label,
            "Scratch_MAE_mean":
                scratch_mae.mean(),
            "Scratch_MAE_std":
                scratch_mae.std(ddof=1),
            "SSL_MAE_mean":
                ssl_mae.mean(),
            "SSL_MAE_std":
                ssl_mae.std(ddof=1),
            "MAE_reduction_percent":
                100.0
                * (
                    scratch_mae.mean()
                    - ssl_mae.mean()
                )
                / scratch_mae.mean(),
        }
    )

gain_table = pd.DataFrame(
    rows
)

print(
    "All four 20/24 runs contain seeds "
    "42, 123, and 2025."
)

display(
    gain_table.round(6)
)

gain_table.to_csv(
    OUT_DIR
    / "linear_ensemble_20_24_ssl_gain_vs_scratch.csv",
    index=False,
    encoding="utf-8-sig",
)

TB3.plot_linear_ensemble_ssl_gain_dotplot_final(
    root=ROOT,
    save_prefix="fig_linear_ensemble_ssl_gain_verified",
)


# **Build direct-alpha dataset from paired ε₁(E) and ε₂(E)**

In [ ]:
import utils.direct_alpha as da

da.HBAR_EV_S = 6.582119569e-16   # eV*s
da.C_LIGHT = 2.99792458e8        # m/s

print("Patched direct_alpha constants")

In [ ]:
alpha_train = TB3.DirectAlphaFromEpsilonDataset(epsR_train_enh_19, epsI_train_enh_19, name="epsI_alpha_log10_1p")
alpha_val   = TB3.DirectAlphaFromEpsilonDataset(epsR_val_enh_19,   epsI_val_enh_19,   name="epsI_alpha_log10_1p")
alpha_test  = TB3.DirectAlphaFromEpsilonDataset(epsR_test_enh_19,  epsI_test_enh_19,  name="epsI_alpha_log10_1p")

sample_alpha = alpha_train[0]

alpha_summary = {
    "train_samples": len(alpha_train),
    "val_samples": len(alpha_val),
    "test_samples": len(alpha_test),
    "node_features": sample_alpha.x.shape[-1],
    "edge_features": sample_alpha.edge_attr.shape[-1],
    "target_shape": tuple(sample_alpha.y.shape),
    "target_min": float(sample_alpha.y.min()),
    "target_max": float(sample_alpha.y.max()),
    "target_mean": float(sample_alpha.y.mean()),
}

assert sample_alpha.x.shape[-1] == 19
assert sample_alpha.edge_attr.shape[-1] == 24
assert sample_alpha.y.shape[-1] == 2001

alpha_summary

# **#  Direct-alpha enhanced GINE scratch baseline**

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_ROOT = Path(r"./")
PROCESSED_DIR = PROJECT_ROOT / "processed"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import utils as TB3

if "device" not in globals():
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

alpha_train_run = alpha_train
alpha_val_run = alpha_val
alpha_test_run = alpha_test

sample_alpha = alpha_train_run[0]

assert sample_alpha.x.shape[-1] == 19
assert sample_alpha.edge_attr.shape[-1] == 24
assert sample_alpha.y.shape[-1] == 2001

alpha_direct_scratch_summary, alpha_direct_scratch_artifacts = (
    TB3.train_optical_gine_single_target(
        target_key="epsI_alpha_log10_1p",
        variant_name="direct_alpha_enhanced_scratch",
        run_name=(
            "epsI_alpha_log10_1p_"
            "direct_alpha_enhanced_scratch_3seed"
        ),
        train_ds=alpha_train_run,
        val_ds=alpha_val_run,
        test_ds=alpha_test_run,
        model_class=TB3.OpticalResponseGINE,
        output_dir=PROCESSED_DIR / "paired_training",
        device=device,
        seeds=(42, 123, 2025),
        batch_size=16,
        num_epochs=100,
        early_stop_patience=15,
        lr=3e-4,
        weight_decay=5e-5,
        hidden_dim=192,
        latent_dim=256,
        num_layers=4,
        dropout=0.05,
        peak_weight=2.0,
        smoothness_weight=0.03,
        pretrained_ckpt_path=None,
    )
)

alpha_direct_scratch_summary


# **Direct-alpha enhanced GINE with SSL initialization**

In [ ]:
from pathlib import Path
import sys
import torch
import utils.direct_alpha as da

PROJECT_ROOT = Path(r"./")
PROCESSED_DIR = PROJECT_ROOT / "processed"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import utils as TB3

# Required until these constants are defined directly in direct_alpha.py.
da.HBAR_EV_S = 6.582119569e-16
da.C_LIGHT = 2.99792458e8

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

alpha_train_run = alpha_train
alpha_val_run = alpha_val
alpha_test_run = alpha_test

SSL_CKPT_ALPHA = Path(
    SSL_CKPT_ENHANCED_BACKBONE
)

assert SSL_CKPT_ALPHA.exists(), SSL_CKPT_ALPHA

sample_alpha = alpha_train_run[0]

assert sample_alpha.x.shape[-1] == 19
assert sample_alpha.edge_attr.shape[-1] == 24
assert sample_alpha.y.shape[-1] == 2001

alpha_direct_ssl_summary, alpha_direct_ssl_artifacts = (
    TB3.train_optical_gine_single_target(
        target_key="epsI_alpha_log10_1p",
        variant_name=(
            "direct_alpha_ssl_init_"
            "enhanced_19_24_d005_wd5e5"
        ),
        run_name=(
            "epsI_alpha_log10_1p_"
            "direct_alpha_ssl_init_"
            "enhanced_19_24_d005_wd5e5_3seed"
        ),
        train_ds=alpha_train_run,
        val_ds=alpha_val_run,
        test_ds=alpha_test_run,
        model_class=TB3.OpticalResponseGINE,
        output_dir=PROCESSED_DIR / "paired_training",
        device=device,
        seeds=(42, 123, 2025),
        batch_size=16,
        num_epochs=100,
        early_stop_patience=15,
        lr=3e-4,
        weight_decay=5e-5,
        hidden_dim=192,
        latent_dim=256,
        num_layers=4,
        dropout=0.05,
        peak_weight=2.0,
        smoothness_weight=0.03,
        pretrained_ckpt_path=SSL_CKPT_ALPHA,
    )
)

alpha_direct_ssl_summary


# _**#  GCN and GraphSAGE baselines on basic 11/16 graph**_

In [ ]:
# GCN and GraphSAGE baselines
# ============================================================

baseline_datasets = {
    "epsI_0": (epsI_train, epsI_val, epsI_test),
    "epsR_0": (epsR_train, epsR_val, epsR_test),
}

baseline_models = {
    "gcn": TB3.OpticalResponseGCN,
    "graphsage": TB3.OpticalResponseGraphSAGE,
}

assert (
    epsI_train[0].x.shape[-1],
    epsI_train[0].edge_attr.shape[-1],
    epsI_train[0].y.numel(),
) == (11, 16, 2001)

baseline_summaries = []

for target_key, datasets in baseline_datasets.items():
    for variant_name, model_class in baseline_models.items():
        summary, _ = TB3.train_optical_gine_single_target(
            target_key=target_key,
            variant_name=variant_name,
            train_ds=datasets[0],
            val_ds=datasets[1],
            test_ds=datasets[2],
            model_class=model_class,
            output_dir=PROCESSED_DIR / "paired_training",
            lr=3e-4,
        )
        baseline_summaries.append(summary)

baseline_summary_table = pd.concat(baseline_summaries, ignore_index=True)
display(baseline_summary_table)

# **Optional enhanced 19/24 GCN and GraphSAGE baselines**

In [ ]:
from pathlib import Path
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

common_baseline_config = dict(
    output_dir=Path(r"D:\TB3\processed\paired_training"),
    device=device,
    seeds=[42, 123, 2025],
    batch_size=16 if device.type == "cuda" else 8,
    num_epochs=100,
    early_stop_patience=15,
    lr=3e-4,
    weight_decay=5e-5,
    hidden_dim=192,
    latent_dim=256,
    num_layers=4,
    dropout=0.05,
    peak_weight=2.0,
    smoothness_weight=0.03,
    pretrained_ckpt_path=None,
)

In [ ]:
import pandas as pd
from IPython.display import display

print("epsI dim:", epsI_train_enh[0].x.shape[1], epsI_train_enh[0].edge_attr.shape[1])
print("epsR dim:", epsR_train_enh[0].x.shape[1], epsR_train_enh[0].edge_attr.shape[1])

assert epsI_train_enh[0].x.shape[1] == 19 and epsI_train_enh[0].edge_attr.shape[1] == 24
assert epsR_train_enh[0].x.shape[1] == 19 and epsR_train_enh[0].edge_attr.shape[1] == 24

enhanced_runs = [
    ("epsI_0", "gcn_enhanced_19_24", TB3.OpticalResponseGCN, epsI_train_enh, epsI_val_enh, epsI_test_enh),
    ("epsI_0", "graphsage_enhanced_19_24", TB3.OpticalResponseGraphSAGE, epsI_train_enh, epsI_val_enh, epsI_test_enh),
    ("epsR_0", "gcn_enhanced_19_24", TB3.OpticalResponseGCN, epsR_train_enh, epsR_val_enh, epsR_test_enh),
    ("epsR_0", "graphsage_enhanced_19_24", TB3.OpticalResponseGraphSAGE, epsR_train_enh, epsR_val_enh, epsR_test_enh),
]

enhanced_baseline_outputs = {}

for target_key, variant_name, model_class, train_ds, val_ds, test_ds in enhanced_runs:
    print("\nRUN:", target_key, "|", variant_name)
    print("dim:", train_ds[0].x.shape[1], train_ds[0].edge_attr.shape[1])

    summary_df, artifacts = TB3.train_optical_gine_single_target(
        target_key=target_key,
        variant_name=variant_name,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        model_class=model_class,
        **common_baseline_config,
    )

    enhanced_baseline_outputs[f"{target_key}_{variant_name}"] = summary_df

# ============================================================
# Merge + save summary
# ============================================================
enhanced_baseline_summary_table = pd.concat(
    list(enhanced_baseline_outputs.values()),
    ignore_index=True
)

display(enhanced_baseline_summary_table)

summary_path = common_baseline_config["output_dir"] / "enhanced_gcn_graphsage_baseline_summary_19_24.csv"
enhanced_baseline_summary_table.to_csv(summary_path, index=False, encoding="utf-8-sig")

print(f"Saved summary to: {summary_path}")

In [ ]:
import pandas as pd
from IPython.display import display

enhanced_baseline_summary_table = pd.concat(
    list(enhanced_baseline_outputs.values()),
    ignore_index=True
)

display(enhanced_baseline_summary_table)

summary_path = common_baseline_config["output_dir"] / "enhanced_gcn_graphsage_baseline_summary_19_24.csv"
enhanced_baseline_summary_table.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Saved:", summary_path)

In [ ]:
mean_table = (
    enhanced_baseline_summary_table
    .groupby(["target_key", "variant"], as_index=False)
    .agg({
        "test_MAE_mean": "mean",
        "test_RMSE_mean": "mean",
        "test_CosineSim_mean": "mean",
        "test_PeakMAE_mean": "mean",
        "test_PeakAxisError_mean": "mean",
    })
)

display(mean_table)

mean_path = common_baseline_config["output_dir"] / "enhanced_gcn_graphsage_baseline_mean_19_24.csv"
mean_table.to_csv(mean_path, index=False, encoding="utf-8-sig")

print("Saved:", mean_path)

# **Compute weighted R² benchmark tables**

In [ ]:
ROOT = PROCESSED_DIR / "paired_training"

r2_tables = TB3.compute_task3_weighted_r2_tables(
    root=ROOT,
    include_gated=True,
    display_tables=True,
)



# _**Final benchmark tables_**

In [ ]:
final_benchmark = TB3.build_task3_final_benchmark_tables(
    root=PROCESSED_DIR / "paired_training",
    display_tables=True,
)

# **Representative ε₂(E) Prediction Examples from SSL-Initialized UE-GINE**

In [ ]:
from pathlib import Path

ROOT = PROCESSED_DIR / "paired_training"
FIG_DIR = ROOT / "final_benchmark_tables" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def single_test_prediction_for_seed(
    folder,
    seed,
):
    matches = sorted(
        Path(folder).glob(
            f"test_predictions_*seed{int(seed)}.npz"
        )
    )

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one test-prediction file for seed {seed} "
            f"in {folder}, found {len(matches)}."
        )

    return matches[0]

epsI_dir = (
    ROOT
    / "epsI_0_ssl_init_enhanced_19_24_d005_wd5e5_3seed"
)

epsI_npz = single_test_prediction_for_seed(
    epsI_dir,
    123,
)

epsI_results = TB3.plot_pred_true_cases(
    npz_path=epsI_npz,
    main_title=(
        r"SSL-init UE-GINE prediction of "
        r"$\varepsilon_2(E)$"
    ),
    y_label=r"$\varepsilon_2(E)$",
    case_mode="paper",
    save_path=(
        FIG_DIR
        / "fig_epsI0_ssl_init_ue_gine_prediction_examples.png"
    ),
)

mae_epsI = epsI_results["mae_each"]
idxs_epsI = epsI_results["selected_indices"]


# **Representative ε₁(E) Prediction Examples from SSL-Initialized UE-GINE**

In [ ]:
epsR_dir = (
    ROOT
    / "epsR_0_ssl_init_enhanced_19_24_d005_wd5e5_3seed"
)

epsR_npz = single_test_prediction_for_seed(
    epsR_dir,
    123,
)

epsR_results = TB3.plot_pred_true_cases(
    npz_path=epsR_npz,
    main_title=(
        r"SSL-init UE-GINE prediction of "
        r"$\varepsilon_1(E)$"
    ),
    y_label=r"$\varepsilon_1(E)$",
    case_mode="paper",
    save_path=(
        FIG_DIR
        / "fig_epsR0_ssl_init_ue_gine_prediction_examples.png"
    ),
)

mae_epsR = epsR_results["mae_each"]
idxs_epsR = epsR_results["selected_indices"]


# **Representative Direct-Alpha Prediction Examples**

In [ ]:
alpha_dir = (
    ROOT
    / (
        "epsI_alpha_log10_1p_"
        "direct_alpha_ssl_init_"
        "enhanced_19_24_d005_wd5e5_3seed"
    )
)

alpha_npz = single_test_prediction_for_seed(
    alpha_dir,
    123,
)

alpha_results = TB3.plot_pred_true_cases(
    npz_path=alpha_npz,
    main_title=(
        r"Direct-alpha ablation on log-scaled "
        r"$\alpha(E)$"
    ),
    y_label=r"$\log_{10}(1+\alpha)$",
    case_mode="paper",
    save_path=(
        FIG_DIR
        / "fig_direct_alpha_ssl_init_ablation_examples.png"
    ),
)

mae_alpha = alpha_results["mae_each"]
idxs_alpha = alpha_results["selected_indices"]


# **Low-Data Evaluation of SSL-Initialized UE-GINE**

In [ ]:
# Train low-data SSL-init vs scratch | enhanced 19/24
import pandas as pd

ROOT = PROCESSED_DIR / "paired_training"
SSL_CKPT = PROCESSED_DIR / "ssl_enhanced_19_24" / "ssl_enhanced_19_24_backbone_only_for_finetune.pt"
OUT_RAW = ROOT / "lowdata_ssl_init_ue_gine_raw.csv"

assert SSL_CKPT.exists(), SSL_CKPT

targets = {
    "epsI_0": (epsI_train_enh, epsI_val_enh, epsI_test_enh),
    "epsR_0": (epsR_train_enh, epsR_val_enh, epsR_test_enh),
}

lowdata_train_kwargs = dict(
    device=device,
    lr=3e-4,
    weight_decay=5e-5,
    batch_size=16,
    num_epochs=100,
    early_stop_patience=15,
    hidden_dim=192,
    latent_dim=256,
    num_layers=4,
    dropout=0.05,
    peak_weight=2.0,
    smoothness_weight=0.03,
)

fractions = [0.05, 0.10, 0.25, 0.50, 1.00]
seeds = [42, 123, 2025]

lowdata_results = {}
raw_tables = []

for fraction in fractions:
    fraction_tag = f"{int(round(fraction * 100))}pct_enhanced19_24"

    result = TB3.run_lowdata_ssl_vs_scratch(
        targets=targets,
        root=ROOT,
        ssl_ckpt=SSL_CKPT,
        label_fraction=fraction,
        fraction_tag=fraction_tag,
        split_seed=777,
        seeds=seeds,
        model_class=TB3.OpticalResponseGINE,
        train_kwargs=lowdata_train_kwargs,
        save_outputs=True,
        show=False,
    )

    lowdata_results[fraction_tag] = result

    df = result["summary"].copy()
    df["label_fraction"] = fraction
    df["fraction_tag"] = fraction_tag
    raw_tables.append(df)

lowdata_raw = pd.concat(raw_tables, ignore_index=True)

if "target_key" not in lowdata_raw.columns and "target" in lowdata_raw.columns:
    lowdata_raw["target_key"] = lowdata_raw["target"]

if "init_type" not in lowdata_raw.columns:
    lowdata_raw["init_type"] = lowdata_raw["variant"].apply(
        lambda x: "ssl_init" if "ssl" in str(x).lower() else "scratch"
    )

lowdata_raw.to_csv(OUT_RAW, index=False, encoding="utf-8-sig")

print("Saved raw table:", OUT_RAW)
print("Rows:", len(lowdata_raw))
display(lowdata_raw.head())

# **Build ready low-data tables**


In [ ]:
import pandas as pd
from IPython.display import display

ROOT = PROCESSED_DIR / "paired_training"

meanstd_files = sorted(ROOT.glob(
    "lowdata_*pct_enhanced19_24_ue_gine_vs_ssl_init_meanstd.csv"
))
gain_files = sorted(ROOT.glob(
    "lowdata_*pct_enhanced19_24_ssl_init_gain.csv"
))

if len(meanstd_files) != 5 or len(gain_files) != 5:
    raise FileNotFoundError(
        f"Expected 5 mean/std and 5 gain files, found "
        f"{len(meanstd_files)} and {len(gain_files)}."
    )

paper_table = pd.concat(
    [pd.read_csv(f) for f in meanstd_files],
    ignore_index=True,
)

gain_table = pd.concat(
    [pd.read_csv(f) for f in gain_files],
    ignore_index=True,
)

paper_table["target_name"] = paper_table["target_key"].map(TB3.get_target_name)
paper_table["model_name"] = paper_table["init_type"].map(TB3.get_model_name)
paper_table["initialization"] = paper_table["model_name"].map(TB3.get_initialization)
paper_table["fraction_label"] = paper_table["label_fraction"].map(TB3.get_fraction_label)

for name, mean_col, std_col in [
    ("MAE", "test_MAE_mean_avg", "test_MAE_mean_std"),
    ("RMSE", "test_RMSE_mean_avg", "test_RMSE_mean_std"),
    ("Cosine", "test_CosineSim_mean_avg", "test_CosineSim_mean_std"),
]:
    paper_table[name] = paper_table.apply(
        lambda r: f"{r[mean_col]:.3f} ± {r[std_col]:.3f}",
        axis=1,
    )

gain_table["target_name"] = gain_table["target_key"].map(TB3.get_target_name)
gain_table["fraction_label"] = gain_table["label_fraction"].map(TB3.get_fraction_label)

paper_table.to_csv(
    ROOT / "table_lowdata_ssl_init_ue_gine_meanstd.csv",
    index=False,
    encoding="utf-8-sig",
)
gain_table.to_csv(
    ROOT / "table_lowdata_ssl_init_ue_gine_gain.csv",
    index=False,
    encoding="utf-8-sig",
)

display(paper_table)
display(gain_table)

# **Low-Data Performance Comparison of UE-GINE and SSL-init UE-GINE**

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

ROOT = PROCESSED_DIR / "paired_training"

wanted = {
    "5pct": "5%",
    "10pct": "10%",
    "25pct": "25%",
    "50pct": "50%",
}

rows = []

for tag, label in wanted.items():
    f = ROOT / f"lowdata_{tag}_enhanced19_24_ue_gine_vs_ssl_init_meanstd.csv"

    if not f.exists():
        print("Missing:", f.name)
        continue

    df = pd.read_csv(f)

    mae_col  = [c for c in df.columns if "test_MAE" in c and "mean" in c][0]
    rmse_col = [c for c in df.columns if "test_RMSE" in c and "mean" in c][0]

    for _, r in df.iterrows():
        variant = str(r["variant"])
        init = "SSL-init" if "ssl_init" in variant else "Scratch"
        target = "ε₁(E)" if r["target_key"] == "epsR_0" else "ε₂(E)"

        rows.append({
            "Target": target,
            "Labeled data": label,
            "Model": "UE-GINE",
            "Initialization": init,
            "MAE": round(r[mae_col], 4),
            "RMSE": round(r[rmse_col], 4),
        })

paper_table_main = pd.DataFrame(rows)

order_frac = {"5%": 0, "10%": 1, "25%": 2, "50%": 3}
order_target = {"ε₁(E)": 0, "ε₂(E)": 1}
order_init = {"Scratch": 0, "SSL-init": 1}

paper_table_main["frac_order"] = paper_table_main["Labeled data"].map(order_frac)
paper_table_main["target_order"] = paper_table_main["Target"].map(order_target)
paper_table_main["init_order"] = paper_table_main["Initialization"].map(order_init)

paper_table_main = (
    paper_table_main
    .sort_values(["frac_order", "target_order", "init_order"])
    .drop(columns=["frac_order", "target_order", "init_order"])
    .reset_index(drop=True)
)

display(paper_table_main)

out_csv = ROOT / "paper_table_lowdata_ssl_init_ue_gine_CLEAN.csv"
paper_table_main.to_csv(out_csv, index=False, encoding="utf-8-sig")

print("Saved:", out_csv)

## Build derived optical-property test pack from fresh dielectric model outputs


In [ ]:
from pathlib import Path
import numpy as np
import utils.screening as SCREEN

ROOT = PROCESSED_DIR / "paired_training"
DERIVED_DIR = (
    ROOT
    / "paper_outputs"
    / "derived_optical_properties"
)

DERIVED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

DERIVED_NPZ = (
    DERIVED_DIR
    / "derived_optical_test_ensemble_ssl_init_ue_gine_100pct.npz"
)

def load_three_seed_ensemble(
    folder,
    expected_seeds=(42, 123, 2025),
):
    folder = Path(
        folder
    )

    pred_list = []
    true_ref = None
    axis_ref = None
    base_idx_ref = None

    for seed in expected_seeds:
        matches = sorted(
            folder.glob(
                f"test_predictions_*seed{seed}.npz"
            )
        )

        if len(matches) != 1:
            raise RuntimeError(
                f"Expected exactly one prediction file for "
                f"seed {seed} in {folder}, found {len(matches)}."
            )

        z = np.load(
            matches[0],
            allow_pickle=True,
        )

        pred = np.asarray(
            z["pred"],
            dtype=np.float64,
        )

        true = np.asarray(
            z["true"],
            dtype=np.float64,
        )

        if pred.ndim == 3 and pred.shape[1] == 1:
            pred = pred[:, 0, :]

        if true.ndim == 3 and true.shape[1] == 1:
            true = true[:, 0, :]

        axis = np.asarray(
            z[
                "spectral_axis"
                if "spectral_axis" in z.files
                else "axis_grid"
            ],
            dtype=np.float64,
        )

        if axis.ndim == 2:
            axis = axis[0]

        axis = axis.reshape(-1)

        base_idx = np.asarray(
            z["base_idx"]
        ).reshape(-1)

        if true_ref is None:
            true_ref = true.copy()
            axis_ref = axis.copy()
            base_idx_ref = base_idx.copy()
        else:
            if not np.allclose(
                true_ref,
                true,
                rtol=1e-6,
                atol=1e-7,
                equal_nan=True,
            ):
                raise RuntimeError(
                    f"Ground truth differs across seeds in {folder}."
                )

            if not np.allclose(
                axis_ref,
                axis,
                rtol=0.0,
                atol=1e-8,
            ):
                raise RuntimeError(
                    f"Energy axis differs across seeds in {folder}."
                )

            if not np.array_equal(
                base_idx_ref,
                base_idx,
            ):
                raise RuntimeError(
                    f"base_idx differs across seeds in {folder}."
                )

        pred_list.append(
            pred
        )

    pred_ensemble = np.mean(
        np.stack(
            pred_list,
            axis=0,
        ),
        axis=0,
    )

    return (
        pred_ensemble,
        true_ref,
        axis_ref,
        base_idx_ref,
    )

# Final dielectric selection used in the paper:
# ε1(E): random-initialized enhanced 19/24 ensemble
# ε2(E): SSL-initialized enhanced 19/24 ensemble
eps1_pred, eps1_true, energy_eV, base_idx = (
    load_three_seed_ensemble(
        ROOT / "epsR_0_enhanced"
    )
)

eps2_pred, eps2_true, energy_eV_2, base_idx_2 = (
    load_three_seed_ensemble(
        ROOT
        / "epsI_0_ssl_init_enhanced_19_24_d005_wd5e5_3seed"
    )
)

if not np.allclose(
    energy_eV,
    energy_eV_2,
    rtol=0.0,
    atol=1e-8,
):
    raise RuntimeError(
        "ε1 and ε2 energy axes do not match."
    )

if not np.array_equal(
    base_idx,
    base_idx_2,
):
    raise RuntimeError(
        "ε1 and ε2 test sample order does not match."
    )

n_true, k_true, alpha_true_cm1, R_true = (
    SCREEN._tb3_screen_dielectric_to_optical(
        eps1_true,
        eps2_true,
        energy_eV,
    )
)

n_pred, k_pred, alpha_pred_cm1, R_pred = (
    SCREEN._tb3_screen_dielectric_to_optical(
        eps1_pred,
        eps2_pred,
        energy_eV,
    )
)

np.savez_compressed(
    DERIVED_NPZ,
    energy_eV=np.asarray(
        energy_eV,
        dtype=np.float32,
    ),
    base_idx=np.asarray(
        base_idx,
    ),
    sample_ids=np.asarray(
        base_idx
    ).astype(str),
    eps1_true=np.asarray(
        eps1_true,
        dtype=np.float32,
    ),
    eps1_pred=np.asarray(
        eps1_pred,
        dtype=np.float32,
    ),
    eps2_true=np.asarray(
        eps2_true,
        dtype=np.float32,
    ),
    eps2_pred=np.asarray(
        eps2_pred,
        dtype=np.float32,
    ),
    n_true=np.asarray(
        n_true,
        dtype=np.float32,
    ),
    n_pred=np.asarray(
        n_pred,
        dtype=np.float32,
    ),
    k_true=np.asarray(
        k_true,
        dtype=np.float32,
    ),
    k_pred=np.asarray(
        k_pred,
        dtype=np.float32,
    ),
    alpha_true_cm1=np.asarray(
        alpha_true_cm1,
        dtype=np.float32,
    ),
    alpha_pred_cm1=np.asarray(
        alpha_pred_cm1,
        dtype=np.float32,
    ),
    R_true=np.asarray(
        R_true,
        dtype=np.float32,
    ),
    R_pred=np.asarray(
        R_pred,
        dtype=np.float32,
    ),
    eps1_model=np.asarray(
        "enhanced_19_24_random_3seed"
    ),
    eps2_model=np.asarray(
        "ssl_init_enhanced_19_24_3seed"
    ),
)

print(
    "Saved fresh derived-optical pack:",
    DERIVED_NPZ,
)


# Representative dielectric-first prediction of optical properties**

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = PROCESSED_DIR / "paired_training"
OUT_DIR = ROOT / "paper_outputs"
FIG_DIR = OUT_DIR / "figures"
DERIVED_DIR = OUT_DIR / "derived_optical_properties"

DERIVED_NPZ = (
    DERIVED_DIR
    / "derived_optical_test_ensemble_ssl_init_ue_gine_100pct.npz"
)

FIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if not DERIVED_NPZ.exists():
    print(
        "Derived optical NPZ was not found. "
        "The fresh-build cell immediately before this figure "
        "should create it."
    )

assert DERIVED_NPZ.exists(), DERIVED_NPZ

data = np.load(
    DERIVED_NPZ,
    allow_pickle=True,
)

E = data["energy_eV"]
sample_ids = data["sample_ids"].astype(str)

preferred_id = "4054"

if preferred_id in sample_ids:
    sample_idx = int(
        np.where(
            sample_ids == preferred_id
        )[0][0]
    )
else:
    err = (
        np.nanmean(
            np.abs(
                data["eps1_pred"]
                - data["eps1_true"]
            ),
            axis=1,
        )
        + np.nanmean(
            np.abs(
                data["eps2_pred"]
                - data["eps2_true"]
            ),
            axis=1,
        )
    )

    sample_idx = int(
        np.argsort(
            err
        )[
            len(err)
            // 2
        ]
    )

print(
    "Selected sample ID:",
    sample_ids[sample_idx],
)

alpha_true = (
    data["alpha_true_cm1"]
    / 1e6
)

alpha_pred = (
    data["alpha_pred_cm1"]
    / 1e6
)

panels = [
    (
        "(a)",
        "Real dielectric function",
        r"$\varepsilon_1(E)$",
        data["eps1_true"][sample_idx],
        data["eps1_pred"][sample_idx],
    ),
    (
        "(b)",
        "Imaginary dielectric function",
        r"$\varepsilon_2(E)$",
        data["eps2_true"][sample_idx],
        data["eps2_pred"][sample_idx],
    ),
    (
        "(c)",
        "Refractive index",
        r"$n(E)$",
        data["n_true"][sample_idx],
        data["n_pred"][sample_idx],
    ),
    (
        "(d)",
        "Extinction coefficient",
        r"$k(E)$",
        data["k_true"][sample_idx],
        data["k_pred"][sample_idx],
    ),
    (
        "(e)",
        "Absorption coefficient",
        r"$\alpha(E)$ ($10^6$ cm$^{-1}$)",
        alpha_true[sample_idx],
        alpha_pred[sample_idx],
    ),
    (
        "(f)",
        "Reflectance",
        r"$R(E)$",
        data["R_true"][sample_idx],
        data["R_pred"][sample_idx],
    ),
]

plt.rcParams.update(
    {
        "font.family": "Arial",
        "mathtext.fontset": "dejavusans",
        "font.size": 12,
        "axes.labelsize": 13,
        "axes.titlesize": 13,
        "legend.fontsize": 11,
    }
)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15.5, 8.2),
    dpi=300,
)

for ax, (
    panel,
    title,
    ylabel,
    y_true,
    y_pred,
) in zip(
    axes.ravel(),
    panels,
):
    ax.plot(
        E,
        y_true,
        lw=2.2,
        label="Reference",
    )

    ax.plot(
        E,
        y_pred,
        lw=2.2,
        ls="--",
        label="Selected GINE ensemble",
    )

    ax.set_title(
        title
    )
    ax.set_xlabel(
        "Energy (eV)"
    )
    ax.set_ylabel(
        ylabel
    )
    ax.grid(
        True,
        alpha=0.25,
    )
    ax.text(
        0.02,
        0.94,
        panel,
        transform=ax.transAxes,
        fontsize=15,
        va="top",
        ha="left",
    )

handles, labels = (
    axes.ravel()[0]
    .get_legend_handles_labels()
)

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 1.02),
)

fig.suptitle(
    "Representative dielectric-first prediction "
    "of optical properties",
    fontsize=16,
    y=1.07,
)

fig.tight_layout(
    rect=[
        0,
        0,
        1,
        0.96,
    ]
)

out_png = (
    FIG_DIR
    / "fig_dielectric_to_optical_prediction.png"
)

out_pdf = (
    FIG_DIR
    / "fig_dielectric_to_optical_prediction.pdf"
)

fig.savefig(
    out_png,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    out_pdf,
    bbox_inches="tight",
)

plt.show()
plt.close(
    fig
)

print("Saved:")
print(out_png)
print(out_pdf)


In [ ]:
# Verify the enhanced graph dimensions used for SSL pretraining

ssl_dims = TB3.check_ssl_graph_dims(
    epsR_train_enh[0],
    expected_node_dim=19,
    expected_edge_dim=24,
)

print(
    f"Training samples: {len(epsR_train_enh)} | "
    f"Node features: {ssl_dims['node_dim']} | "
    f"Edge features: {ssl_dims['edge_dim']} | "
    f"SSL output dimension: {ssl_dims['ssl_out_dim']}"
)

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path(r"D:\TB3\processed\paired_training")

summary_files = sorted(
    ROOT.glob("lowdata_*pct_enhanced19_24_ue_gine_vs_ssl_init_summary.csv")
)

if len(summary_files) != 5:
    raise FileNotFoundError(
        f"Expected 5 low-data summary files, found {len(summary_files)}."
    )

cols = [
    "target_key",
    "init_type",
    "seed",
    "label_fraction",
    "test_MAE_mean",
]

raw = (
    pd.concat(
        [pd.read_csv(path)[cols] for path in summary_files],
        ignore_index=True,
    )
    .drop_duplicates(
        ["target_key", "init_type", "seed", "label_fraction"]
    )
)

figure3_stats = (
    raw.groupby(
        ["target_key", "init_type", "label_fraction"],
        as_index=False,
    )
    .agg(
        n_seeds=("seed", "nunique"),
        test_MAE_mean=("test_MAE_mean", "mean"),
        test_MAE_std=("test_MAE_mean", lambda x: x.std(ddof=1)),
    )
    .sort_values(["target_key", "init_type", "label_fraction"])
    .reset_index(drop=True)
)

if not figure3_stats["n_seeds"].eq(3).all():
    raise RuntimeError("Each Figure 3 point must contain exactly three seeds.")

OUT_CSV = ROOT / "figure3_label_efficiency_meanstd_ddof1.csv"
figure3_stats.to_csv(OUT_CSV, index=False)

display(figure3_stats)
print(f"Saved: {OUT_CSV}")

## Random-split overlap and generalization audit

In [ ]:
from pathlib import Path
import importlib
import utils as TB3
import utils.generalization as GEN

print("utils:", TB3.__file__)

UTILS_DIR = Path(TB3.__file__).parent
assert (UTILS_DIR / "generalization.py").exists(), (
    f"Missing: {UTILS_DIR / 'generalization.py'}"
)

importlib.reload(GEN)

split_audit = GEN.run_split_generalization_audit(
    project_root=PROJECT_ROOT,
    dataset=dataset,
)

In [ ]:
# Matched full-data comparison: Scratch vs SSL-init
# Enhanced graph representation: 19 node / 24 edge features

from pathlib import Path
import importlib
from IPython.display import display

import utils as TB3
import utils.features as FEAT

FEAT = importlib.reload(FEAT)

ROOT = PROCESSED_DIR / "paired_training"

SSL_CKPT = (
    PROCESSED_DIR
    / "ssl_enhanced_19_24"
    / "ssl_enhanced_19_24_backbone_only_for_finetune.pt"
)

assert SSL_CKPT.exists(), SSL_CKPT

# ============================================================
# Enhanced graph datasets
# ============================================================

epsR_graph, epsI_graph, *_ = FEAT.build_paired_dielectric_graph_datasets_enhanced(
    dataset,
    key_real="epsR_0",
    key_imag="epsI_0",
    verbose=False,
)

print("=" * 100)
print("ENHANCED GRAPH CHECK")
print("=" * 100)
print(
    "epsR_0:",
    epsR_graph[0].x.shape[-1],
    "node features /",
    epsR_graph[0].edge_attr.shape[-1],
    "edge features",
)
print(
    "epsI_0:",
    epsI_graph[0].x.shape[-1],
    "node features /",
    epsI_graph[0].edge_attr.shape[-1],
    "edge features",
)

assert (
    epsR_graph[0].x.shape[-1],
    epsR_graph[0].edge_attr.shape[-1],
) == (19, 24)

assert (
    epsI_graph[0].x.shape[-1],
    epsI_graph[0].edge_attr.shape[-1],
) == (19, 24)

# Fixed paired split
# ============================================================

split = FEAT.build_fixed_paired_split(
    epsR_graph,
    epsI_graph,
    processed_dir=PROCESSED_DIR,
    seed=2025,
    overwrite=False,
)

targets = {
    "epsR_0": (
        split["epsR_train_ds"],
        split["epsR_val_ds"],
        split["epsR_test_ds"],
    ),
    "epsI_0": (
        split["epsI_train_ds"],
        split["epsI_val_ds"],
        split["epsI_test_ds"],
    ),
}

print("=" * 100)
print("FIXED DATASET SIZES")
print("=" * 100)

for target_key, (train_ds, val_ds, test_ds) in targets.items():
    print(
        f"{target_key}: "
        f"train={len(train_ds)}, "
        f"val={len(val_ds)}, "
        f"test={len(test_ds)}"
    )
# ============================================================
# Matched Scratch vs SSL-init training
# ============================================================

results_matched_full = TB3.run_lowdata_ssl_vs_scratch(
    targets=targets,
    root=ROOT,
    ssl_ckpt=SSL_CKPT,
    label_fraction=1.0,
    fraction_tag="100pct_matched_full_rerun_enhanced19_24",
    split_seed=777,
    seeds=(42, 123, 2025),
    train_kwargs={"lr": 3e-4},
    save_outputs=True,
    show=False,
)

# ============================================================
# Final tables
# ============================================================

print("\n" + "=" * 100)
print("PER-SEED RESULTS")
print("=" * 100)
display(results_matched_full["summary"])

print("\n" + "=" * 100)
print("MEAN AND STANDARD DEVIATION")
print("=" * 100)
display(results_matched_full["meanstd"])

print("\n" + "=" * 100)
print("SSL GAIN RELATIVE TO SCRATCH")
print("=" * 100)
display(results_matched_full["gain"])

# **## Limited-Label Repeated-Subset Robustness Test**

In [ ]:
if RUN_REPEATED_SUBSET_ROBUSTNESS:
    import importlib
    import utils.robustness as ROB

    ROB = importlib.reload(ROB)

    repeated_subset_results = (
        ROB.run_repeated_subset_robustness(
            project_root=PROJECT_ROOT,
            label_fractions=(0.25, 0.50),
            subset_seeds=(777, 2026, 3407),
            model_seeds=(42, 123, 2025),
            force_rerun=True,
            show=True,
        )
    )
else:
    print(
        "Skipped repeated-subset robustness. "
        "Set RUN_REPEATED_SUBSET_ROBUSTNESS = True "
        "in the first cell to run the full 72-run analysis."
    )


## Optional external DFT validation (requires external DFT outputs)


In [ ]:
DFT_SOURCE = Path(r"D:\DFT_TEST")

if RUN_EXTERNAL_DFT_VALIDATION and DFT_SOURCE.exists():
    import importlib
    import utils.dft_validation as DV

    DV = importlib.reload(DV)

    dft_validation = DV.run_dft_validation_figure(
        project_root=PROJECT_ROOT,
        dft_source=DFT_SOURCE,
        show=True,
    )
else:
    print(
        "Skipped legacy external DFT validation. "
        "It requires D:\\DFT_TEST and is not generated "
        "from database_300.zip."
    )


In [ ]:
# Direct-alpha GNNOpt-style held-out test
# ============================================================

import importlib
import utils.direct_alpha_eval as DAE

DAE = importlib.reload(DAE)

direct_alpha_eval = DAE.run_gnnopt_style_direct_alpha_test(
    project_root=r"D:\TB3",
    show=True,
)

In [ ]:
if RUN_POST_SCREENING_AUDITS:
    import importlib
    import utils.screening_audit as SA

    SA = importlib.reload(SA)

    screening_audit = SA.audit_final_screening_inputs(
        project_root=PROJECT_ROOT,
    )
else:
    print(
        "Skipped screening-input audit. "
        "Run the external-screening notebook first, then "
        "set RUN_POST_SCREENING_AUDITS = True if required."
    )


In [ ]:
if RUN_POST_SCREENING_AUDITS:
    import importlib
    import utils.screening_audit as SA

    SA = importlib.reload(SA)

    clean_alpha_forensics = SA.audit_clean_alpha_forensics(
        project_root=PROJECT_ROOT,
    )
else:
    print(
        "Skipped clean-alpha forensic audit because "
        "external-screening outputs are not part of the "
        "main raw-data training notebook."
    )


In [ ]:
if RUN_POST_SCREENING_AUDITS:
    import importlib
    import utils.slme_provenance as SP

    SP = importlib.reload(SP)

    slme_source_audit = SP.find_exact_slme_implementation(
        project_root=PROJECT_ROOT,
    )
else:
    print(
        "Skipped SLME provenance scan. "
        "The final SLME implementation is reproduced "
        "in the downstream held-out and screening notebooks."
    )


In [ ]:
# ============================================================
# Verify SLME validation utility
# ============================================================

from pathlib import Path
import utils

MODULE_PATH = Path(r"D:\TB3\utils_tb\slme_validation.py")

print("utils_tb loaded from :", utils_tb.__file__)
print("Expected module path :", MODULE_PATH)
print("Module exists        :", MODULE_PATH.exists())

In [ ]:
DFT_ZIP = Path(
    r"D:\DFT_TEST\collect_data_epsilon.zip"
)

if RUN_EXTERNAL_DFT_VALIDATION and DFT_ZIP.exists():
    import importlib
    import utils.slme_validation as SV

    SV = importlib.reload(SV)

    slme_validation = (
        SV.run_final_dft22_slme_validation(
            project_root=PROJECT_ROOT,
            dft_zip=DFT_ZIP,
            show=True,
        )
    )
else:
    print(
        "Skipped 22-sample DFT SLME validation. "
        "This requires the external DFT archive "
        "collect_data_epsilon.zip."
    )


In [ ]:
DFT_ZIP = Path(
    r"D:\DFT_TEST\collect_data_epsilon.zip"
)

if RUN_EXTERNAL_DFT_VALIDATION and DFT_ZIP.exists():
    import importlib
    import utils.supplementary_spectral_validation as SSV

    SSV = importlib.reload(SSV)

    supplementary_validation = (
        SSV.run_supplementary_full22_spectral_validation(
            project_root=PROJECT_ROOT,
            dft_zip=DFT_ZIP,
            show=True,
        )
    )
else:
    print(
        "Skipped supplementary 22-sample spectral validation. "
        "External DFT data are not generated from database_300.zip."
    )


In [ ]:
DFT_ZIP = Path(
    r"D:\DFT_TEST\collect_data_epsilon.zip"
)

if RUN_EXTERNAL_DFT_VALIDATION and DFT_ZIP.exists():
    import importlib
    import utils.main_figure_cd as MFCD

    MFCD = importlib.reload(MFCD)

    main_figure_cd = MFCD.run_final_main_figure_cd(
        project_root=PROJECT_ROOT,
        dft_zip=DFT_ZIP,
        show=True,
    )
else:
    print(
        "Skipped final DFT validation figure panels. "
        "They require the external DFT archive."
    )


## Final fresh-build validation


In [ ]:
# ============================================================
# Final fresh-build check
# ============================================================

required_runs = {
    "epsI primary SSL":
        TRAINING_DIR
        / "epsI_0_ssl_init_enhanced_19_24_d005_wd5e5_3seed",

    "epsR primary SSL":
        TRAINING_DIR
        / "epsR_0_ssl_init_enhanced_19_24_d005_wd5e5_3seed",

    "Direct-alpha SSL":
        TRAINING_DIR
        / (
            "epsI_alpha_log10_1p_"
            "direct_alpha_ssl_init_"
            "enhanced_19_24_d005_wd5e5_3seed"
        ),

    "Contribution epsI 20/24":
        TRAINING_DIR
        / "epsI_0_ssl_init_linear_ue_20_24",

    "Contribution epsR 20/24":
        TRAINING_DIR
        / "epsR_0_ssl_init_linear_ue_20_24",
}

check_rows = []

for name, folder in required_runs.items():
    checkpoints = sorted(
        folder.glob(
            "best_optical_*seed*.pt"
        )
    )

    predictions = sorted(
        folder.glob(
            "test_predictions_*seed*.npz"
        )
    )

    check_rows.append(
        {
            "run": name,
            "folder": str(folder),
            "exists": folder.exists(),
            "n_checkpoints": len(checkpoints),
            "n_test_predictions": len(predictions),
            "pass": (
                folder.exists()
                and len(checkpoints) == 3
                and len(predictions) == 3
            ),
        }
    )

fresh_build_check = pd.DataFrame(
    check_rows
)

display(
    fresh_build_check
)

if not fresh_build_check["pass"].all():
    raise RuntimeError(
        "Fresh-build check failed. "
        "Do not continue to the downstream SLME notebook."
    )

print("=" * 100)
print("MAIN FRESH BUILD COMPLETE")
print("=" * 100)
print(
    "The core processed/ outputs were generated from "
    "database_300.zip and utils_tb."
)
print()
print(
    "Next: 05_direct_alpha_SLME_500nm_heldout_FINAL.ipynb"
)
print(
    "Then: 01_build_zintl_like_pool_no_element_filter_"
    "DISCOVERY_READY_DB2_FIXED.ipynb"
)
